# Validate Monthly Power BI Snapshot

## Purpose

This notebook performs an initial validation of one synthetic monthly snapshot generated in the previous notebook.

The aim is to confirm that the exported CSV file loads successfully, contains the expected fields and appears suitable for the full exploratory analysis completed in the next notebook.

---

## Validation Checks

This notebook:

- Loads one monthly CSV snapshot
- Confirms the dataset dimensions
- Reviews the column names and data types
- Previews a sample of records
- Checks for obvious missing values or formatting problems

Detailed trend and business analysis is completed in `03_exploratory_analysis.ipynb`.

In [1]:
import pandas as pd

## Load the Dataset

The monthly snapshot is loaded into a Pandas DataFrame to begin the exploratory data analysis process.

In [2]:
# Load the Power BI snapshot

df = pd.read_csv("../data/raw/powerbi_snapshot_2026_01.csv")

## Preview the Dataset

The first few records are displayed to confirm that the dataset has loaded correctly and to understand the available fields before further exploration.

In [3]:
df.head()

,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Agent Working,Refund Type,Root Cause,Customer Contacted,Outcome
0,DH001911,PET001911,CL000004,2026-01-31,29.94,Void,Retentions,RET009,DH001,Cancellation from Renewal (Void),Agent Forgot,No,Actioned
1,DH001912,PET001912,CL000036,2026-01-31,71.62,Void,Retentions,RET021,DH001,Cancellation from Renewal (Void),Refund Mailbox Delay,No,Actioned
2,DH001913,PET001913,CL000196,2026-01-31,5.45,Void,Claims,CLM008,DH001,Cancellation from Renewal (Void),Payment Date Misunderstood,No,Actioned
3,DH001914,PET001914,CL000113,2026-01-31,73.56,Void,Customer Service,CS004,DH002,Death of Pet,Waiting for Information,No,Actioned
4,DH001915,PET001915,CL000185,2026-01-31,10.16,Cancelled,Claims,CLM006,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,Actioned


## Structural Validation

The snapshot is checked for its dimensions, missing values and duplicate case identifiers.

In [4]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate Case IDs: {df['Case ID'].duplicated().sum()}")

Rows: 191
Columns: 13
Missing values: 0
Duplicate Case IDs: 0


In [5]:
expected_columns = {
    "Case ID",
    "Policy Number",
    "Client Number",
    "Snapshot Date",
    "Outstanding Amount",
    "Cancellation Status",
    "Cancelling Department",
    "Cancelling Agent",
    "Agent Working",
    "Refund Type",
    "Root Cause",
    "Customer Contacted",
    "Outcome",
}

actual_columns = set(df.columns)

print("Missing columns:", expected_columns - actual_columns)
print("Unexpected columns:", actual_columns - expected_columns)

Missing columns: set()
Unexpected columns: set()


## Business-Rule Validation

The snapshot is checked against the documented amount and cancellation-status rules.

In [6]:
invalid_amounts = ~df["Outstanding Amount"].between(
    0.50,
    300.00,
)

invalid_renewal_status = (
    (df["Refund Type"] == "Cancellation from Renewal (Void)")
    & (df["Cancellation Status"] != "Void")
)

invalid_before_premium_status = (
    (df["Refund Type"] == "Cancellation Before Premium Due")
    & (df["Cancellation Status"] != "Cancelled")
)

print("Invalid outstanding amounts:", invalid_amounts.sum())
print("Invalid renewal statuses:", invalid_renewal_status.sum())
print(
    "Invalid before-premium statuses:",
    invalid_before_premium_status.sum(),
)

Invalid outstanding amounts: 0
Invalid renewal statuses: 0
Invalid before-premium statuses: 0


### Observation

The dataset loaded successfully and contains the expected operational fields required for analysis.

No obvious formatting issues are visible within the first few records.

## Validation Summary

The January 2026 snapshot passed the structural and business-rule checks:

- 191 rows and 13 expected columns
- no missing values
- no duplicate case IDs
- no missing or unexpected columns
- all outstanding amounts within the £0.50–£300.00 range
- all renewal cancellations correctly recorded as `Void`
- all before-premium-due cancellations correctly recorded as `Cancelled`

The validated monthly files are suitable for consolidation and further exploratory analysis.